#Preparando Ambiente

In [1]:
# Downloads necessários
!pip install transformers
!pip install einops accelerate bitsandbytes
!pip install sentence_transformers
!pip install git+https://github.com/huggingface/peft.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 41.3 MB/s eta 0:00:00
  Cloning https://github.com/huggingface/peft.git to /tmp/pip-req-build-e76xzom6
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/peft.git /tmp/pip-req-build-e76xzom6
  Resolved https://github.com/huggingface/peft.git to commit 261366de2e40cde64b702d6b9c527081ad850549
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for peft: filename=peft-0.18.1.dev0-py3-none-any.whl size=569869 sha256=f5593a99377b7700a702b425bee937cd59a636f636236e7834ae26bf186fd94a
  Stored in directory: /tmp/pip-ephem-wheel-cache-liiqjhtm/wheels/5d/16/61/117d50be36b7cb532817817523554825ff840d223c0f65c2c4
Successfully built peft
  Attempting uninstall: peft
    Found existing installation: peft 0.18.0
    Uninstalling peft-0.18.0:
      Successfully uninstalled peft-0.18.0


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
from peft import PeftModel, PeftConfig
from datetime import datetime
from zoneinfo import ZoneInfo
import pandas as pd
import random
import torch
import re

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Carregando dataset ASSIN2

In [4]:
splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet', 'validation': 'data/validation-00000-of-00001.parquet'}
df_assin_2_treino = pd.read_parquet("hf://datasets/nilc-nlp/assin2/" + splits["train"])
df_assin_2_teste = pd.read_parquet("hf://datasets/nilc-nlp/assin2/" + splits["test"])
df_assin_2_val = pd.read_parquet("hf://datasets/nilc-nlp/assin2/" + splits["validation"])

df_assin_2 = pd.concat([df_assin_2_treino, df_assin_2_teste, df_assin_2_val])

#Carregando Bode 7B

In [5]:
config = PeftConfig.from_pretrained('recogna-nlp/bode-7b-alpaca-pt-br')
model = AutoModelForCausalLM.from_pretrained(config.base_model_name_or_path, trust_remote_code=True, return_dict=True, device_map='auto')

tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)
model = PeftModel.from_pretrained(model, 'recogna-nlp/bode-7b-alpaca-pt-br', offload_folder="/offload_dir")

adapter_config.json:   0%|          | 0.00/451 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

adapter_model.bin:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

In [6]:
def zero_shot_prompt(premise, hypothesis):
  return f"""
   Você é um sistema de Reconhecimento de Inferência Textual (RTE) em Português Brasileiro.

    Tarefa:
    Dada uma PREMISSA e uma HIPÓTESE, responda *apenas* com um único caractere:
    - 0 se a hipótese não é inferida da premissa.
    - 1 se a hipótese é logicamente inferida da premissa.

    Regras obrigatórias:
    - NÃO explique.
    - NÃO acrescente texto.
    - NÃO repita o enunciado.
    - NÃO responda nada além de 0 ou 1.
    """

def few_shot_prompt(premise, hypothesis):
  return f"""Você é um avaliador de inferência textual em Português. Para cada par (Premissa / Hipótese), responda apenas uma palavra: "Implica" ou "Não Implica".
  Baseie-se no que pode ser logicamente inferido, levando em conta implicaturas e contexto.

  Exemplo 1
  Premissa: "Um cachorro está latindo no quintal."
  Hipótese: "Há um cachorro no quintal."
  Resposta: Implica

  Exemplo 2
  Premissa: "Uma pessoa correu para a escola."
  Hipótese: "A pessoa perdeu o ônibus."
  Resposta: Não Implica

  Exemplo 3
  Premissa: "O carro parou porque o motor falhou."
  Hipótese: "O carro quebrou."
  Resposta: Implica

  Agora avalie:
  Premissa: {premise}
  Hipótese: {hypothesis}
  Resposta:
  """

In [7]:
num_records = len(df_assin_2)
random_index = random.randint(0, num_records - 1)
random_index = 3

premissa = df_assin_2.iloc[random_index]['premise']
hipotese = df_assin_2.iloc[random_index]['hypothesis']
prompt = zero_shot_prompt(premissa, hipotese)


inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=10)
prompt_len = inputs["input_ids"].shape[1]
generated_tokens = output[0][prompt_len:]
resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print(f'Prompt: {prompt}')
print(f'Resposta: {resp}')

Prompt: 
   Você é um sistema de Reconhecimento de Inferência Textual (RTE) em Português Brasileiro.

    Tarefa:
    Dada uma PREMISSA e uma HIPÓTESE, responda *apenas* com um único caractere:
    - 0 se a hipótese não é inferida da premissa.
    - 1 se a hipótese é logicamente inferida da premissa.

    Regras obrigatórias:
    - NÃO explique.
    - NÃO acrescente texto.
    - NÃO repita o enunciado.
    - NÃO responda nada além de 0 ou 1.
    
Resposta: 
    Exemplo:
    PREMI


In [8]:
for i in range(len(df_assin_2)):
  premissa = df_assin_2.iloc[i]['premise']
  hipotese = df_assin_2.iloc[i]['hypothesis']

  prompt = zero_shot_prompt(premissa, hipotese)

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  output = model.generate(**inputs, max_new_tokens=10)
  prompt_len = inputs["input_ids"].shape[1]
  generated_tokens = output[0][prompt_len:]
  resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)

  df_assin_2.loc[i, 'bode_7b'] = resp

  if i%100==0:
    print(f"{i} - {datetime.now(ZoneInfo("America/Sao_Paulo")).strftime("%Y-%m-%d %H:%M:%S")}")

df_assin_2.to_csv('/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_bode_7b.csv')

0 - 2025-12-22 23:31:51
100 - 2025-12-22 23:32:45
200 - 2025-12-22 23:33:39
300 - 2025-12-22 23:34:34
400 - 2025-12-22 23:35:29
500 - 2025-12-22 23:36:24
600 - 2025-12-22 23:37:19
700 - 2025-12-22 23:38:14
800 - 2025-12-22 23:39:07
900 - 2025-12-22 23:40:01
1000 - 2025-12-22 23:40:55
1100 - 2025-12-22 23:41:49
1200 - 2025-12-22 23:42:43
1300 - 2025-12-22 23:43:37
1400 - 2025-12-22 23:44:31
1500 - 2025-12-22 23:45:25
1600 - 2025-12-22 23:46:19
1700 - 2025-12-22 23:47:13
1800 - 2025-12-22 23:48:07
1900 - 2025-12-22 23:49:01
2000 - 2025-12-22 23:49:55
2100 - 2025-12-22 23:50:49
2200 - 2025-12-22 23:51:43
2300 - 2025-12-22 23:52:37
2400 - 2025-12-22 23:53:31
2500 - 2025-12-22 23:54:25
2600 - 2025-12-22 23:55:18
2700 - 2025-12-22 23:56:12
2800 - 2025-12-22 23:57:06
2900 - 2025-12-22 23:57:59
3000 - 2025-12-22 23:58:53
3100 - 2025-12-22 23:59:47
3200 - 2025-12-23 00:00:40
3300 - 2025-12-23 00:01:34
3400 - 2025-12-23 00:02:28
3500 - 2025-12-23 00:03:22
3600 - 2025-12-23 00:04:15
3700 - 2025-1